In [4]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
import streamlit as st
from dotenv import load_dotenv
load_dotenv()
#-----------------------------
model1=ChatGoogleGenerativeAI(model='gemini-2.5-flash')
model2=ChatOpenAI(model='gpt-3.5-turbo')
#-----------------------------
prompt1=PromptTemplate(template='Generate a summary on this {Topic}', input_variables=['Topic'])
prompt2=PromptTemplate(template='Generate 5 quiz question on {Topic}',input_variables=['Topic'])
#-----------------------------
parser=StrOutputParser()
#-----------------------------
parallel_chain=RunnableParallel({'summary': prompt1 | model1 | parser, 'quiz': prompt1 | model2 | parser})
prompt3=PromptTemplate(template='Merge the following {summary} and {quiz} together', input_variables=['summary','quiz'])
#-----------------------------
chain = prompt3 | model2 | parser
final_chain = parallel_chain | chain 
#-----------------------------
result=final_chain.invoke({'Topic':'Agent AI'})
result

'Agent AI is a powerful tool that uses artificial intelligence to assist customer service agents in providing personalized and efficient support to customers. It can analyze customer inquiries, suggest responses, and even automate certain tasks, ultimately saving time and ensuring a more streamlined customer service experience. Agent AI is designed to enhance the overall effectiveness and efficiency of customer service operations by leveraging the capabilities of AI technology. This technology merges the autonomy, perception, reasoning/decision-making, action, goal-oriented, and persistence characteristics of Agent AI into a system that supports and enhances human agents in their interactions with customers.'

In [ ]:
final_chain.get_graph().print_ascii()

In [5]:
## Conditioned CHAIN

In [10]:
from pydantic import Field, BaseModel
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableBranch, RunnableLambda
from dotenv import load_dotenv
load_dotenv()
#----------------------------------------
class Sentiment(BaseModel):
    sentiment: Literal['Positive','Negative']=Field(description='Generate the sentiment either positive or negative')
parser=PydanticOutputParser(pydantic_object=Sentiment)
prompt1=PromptTemplate(template='Generate the sentiment of the following {feedback} is {format}',
                       input_variables=['feedback'],partial_variables={'format':parser.get_format_instructions()})
model1=ChatOpenAI(model='gpt-3.5-turbo')
model2=ChatGoogleGenerativeAI(model='gemini-2.5-flash')
strparser=StrOutputParser()

chainS= prompt1 | model1 | parser
#chainS.invoke({'i like this product'}).sentiment

prompt2=PromptTemplate(template='Generate the positive response for the received {Sentiment} to user', input_variables=['Sentiment'])
prompt3=PromptTemplate(template='Generate the response as SORRY message for the received {Sentiment} to user', input_variables=['Sentiment'])

conditional_chain=RunnableBranch((lambda x:x.sentiment=='Positive', prompt2 | model2 | strparser),
                                 (lambda x:x.sentiment=='Negative', prompt3 | model2 | strparser),
                                 RunnableLambda(lambda x:'could not extract the sentiment'))
final_chain = chainS | conditional_chain
response=final_chain.invoke("Iphone 12 is not good")
print(response)

I'm truly sorry to hear that. It sounds like you're experiencing something negative, and I'm very sorry you're feeling that way.

Is there anything I can do to help, or something you'd like to talk about? Please let me know.
